In [1]:
# from jobanalysis.io import load
# from jobanalysis.paths import FINAL_DATA, SAMPLE_DATA

In [2]:
# df = load(FINAL_DATA)

# df.info()

In [3]:
# df = df.sample(n=150, random_state=42)

# df.info()

In [4]:
# df.to_csv(SAMPLE_DATA, index=False)

In [5]:
from jobanalysis.paths import LABELED_DATA
from sklearn.metrics import classification_report, precision_score, recall_score
from statsmodels.stats.proportion import proportion_confint
import pandas as pd

df = pd.read_csv(LABELED_DATA)
df.head()

,title,company,salary,category,created_at,deadline_at,view_count,description,experience_years_min,education_level_min,required_languages,experience_years_min_label,education_level_min_label,required_languages_label
0,Data Analyst,IRES,1500.0,Komputerləşmə və İKT,2026-04-22 00:00:00+04:00,2026-05-22 00:00:00+04:00,1800,Job Overview: \n\n \n\t We are currently looki...,1.0,Bachelor,English,1.0,Bachelor,English
1,Brend menecer,Prime Accessories Mağazalar Şəbəkəsi,NaN,"Marketinq, reklam, çap və nəşriyyat",2026-04-17 00:00:00+04:00,2026-05-17 00:00:00+04:00,514,BİZ SADƏCƏ SATIŞ ÜZRƏ BREND MENECER AXTARIŞIND...,3.0,Bachelor,"Azerbaijani,English,Russian",3.0,Bachelor,"Azerbaijani,Russian,English"
2,Maddi vəsaitlər üzrə aparıcı mütəxəssis (Xudat...,Foodcity Agropark,NaN,"Nəqliyyat, paylama və logistika",2026-04-27 00:00:00+04:00,2026-05-27 00:00:00+04:00,131,"Tələblər: \n\n \n\t Ali təhsil (logistika, təc...",3.0,Bachelor,Azerbaijani,3.0,Bachelor,Azerbaijani
3,Uşaqlar üçün ailə Sürücüsü,Massi,1200.0,"Nəqliyyat, paylama və logistika",2026-06-22 00:00:00+04:00,2026-07-22 00:00:00+04:00,1700,Namizədə tələblər \n\n \n\t Xanım namizəd \n\t...,3.0,Not Mentioned,Azerbaijani,3.0,Not Mentioned,Azerbaijani
4,Merçendayzer,Fostanpak MMC,700.0,Pərakəndə satış və müştəri xidmətləri,2026-06-23 00:00:00+04:00,2026-07-23 00:00:00+04:00,127,Vəzifə öhdəlikləri \n\n \n\t Malların vitrində...,NaN,Not Mentioned,Azerbaijani,NaN,Not Mentioned,Azerbaijani


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 14 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   title                       150 non-null    str    
 1   company                     145 non-null    str    
 2   salary                      43 non-null     float64
 3   category                    150 non-null    str    
 4   created_at                  150 non-null    str    
 5   deadline_at                 150 non-null    str    
 6   view_count                  150 non-null    int64  
 7   description                 150 non-null    str    
 8   experience_years_min        101 non-null    float64
 9   education_level_min         150 non-null    str    
 10  required_languages          150 non-null    str    
 11  experience_years_min_label  106 non-null    float64
 12  education_level_min_label   150 non-null    str    
 13  required_languages_label    150 non-null    st

In [7]:
(df["required_languages"] != df["required_languages_label"]).sum()

np.int64(35)

In [8]:
df["education_level_min"].value_counts()

education_level_min
Bachelor         88
Not Mentioned    43
Middle           19
Name: count, dtype: int64

In [9]:
(df["education_level_min"] != df["education_level_min_label"]).sum()

np.int64(3)

In [10]:
y_true = df["education_level_min"]
y_pred = df["education_level_min_label"]

print(classification_report(y_true, y_pred, digits=3))

               precision    recall  f1-score   support

     Bachelor      0.989     0.989     0.989        88
       Middle      0.944     0.895     0.919        19
Not Mentioned      0.977     1.000     0.989        43

     accuracy                          0.980       150
    macro avg      0.970     0.961     0.965       150
 weighted avg      0.980     0.980     0.980       150



In [11]:
for label in y_true.unique():
    precision = precision_score(y_true, y_pred, labels=[label], average="micro")
    recall = recall_score(y_true, y_pred, labels=[label], average="micro")

    true_count = (y_true == label).sum()
    pred_count = (y_pred == label).sum()
    correct = ((y_true == label) & (y_pred == label)).sum()

    recall_ci = proportion_confint(correct, true_count, method="wilson")
    precision_ci = proportion_confint(correct, pred_count, method="wilson")

    print(f"\n{label}")
    print(f"  precision: {precision:.3f}  CI: {precision_ci}")
    print(f"  recall:    {recall:.3f}  CI: {recall_ci}")


Bachelor
  precision: 0.989  CI: (0.9384050610770144, 0.9979912146822955)
  recall:    0.989  CI: (0.9384050610770144, 0.9979912146822955)

Not Mentioned
  precision: 0.977  CI: (0.8819229030178671, 0.9959767479399467)
  recall:    1.000  CI: (0.9179901967742088, 1.0)

Middle
  precision: 0.944  CI: (0.7424269921812741, 0.990124809021697)
  recall:    0.895  CI: (0.6860591728889099, 0.9706414397011299)


In [ ]:
exp = df.dropna(subset=["experience_years_min", "experience_years_min_label"])

print(f"Total rows:                              {len(df)}")
print(f"Extractor found a value:                 {df['experience_years_min'].notna().sum()}")
print(f"Label says experience is mentioned:      {df['experience_years_min_label'].notna().sum()}")
print(f"Both have a value (used for comparison): {len(exp)}")

Total rows:                              150
Extractor found a value:                 101
Label says experience is mentioned:      106
Both have a value (used for comparison): 99


In [13]:
(exp["experience_years_min"] != exp["experience_years_min_label"]).sum()

np.int64(1)

In [ ]:
exact      = (exp["experience_years_min"] == exp["experience_years_min_label"]).sum()
exact_rate = exact / len(exp)
exact_ci   = proportion_confint(exact, len(exp), method="wilson")

# MAE = average absolute difference in years between our prediction and the label
mae = abs(exp["experience_years_min"] - exp["experience_years_min_label"]).mean()

print(f"Exact match rate: {exact_rate:.3f}  CI: {exact_ci}")
print(f"Mean Absolute Error (years): {mae:.3f}")

Exact match rate: 0.990  CI: (0.9449846853541273, 0.9982146927208302)
Mean Absolute Error (years): 0.040


In [ ]:
pred  = df["required_languages"]
label = df["required_languages_label"]

(pred != label).sum()

np.int64(35)

In [ ]:
LANGUAGES = ["Azerbaijani", "English", "Russian"]

def parse_langs(val):
    if pd.isna(val):
        return set()
    return set(str(val).split(","))

y_true_lang = pd.DataFrame(
    {lang: df["required_languages_label"].apply(lambda v: lang in parse_langs(v)) for lang in LANGUAGES}
)
y_pred_lang = pd.DataFrame(
    {lang: df["required_languages"].apply(lambda v: lang in parse_langs(v)) for lang in LANGUAGES}
)

print(classification_report(y_true_lang, y_pred_lang, target_names=LANGUAGES, digits=3))

              precision    recall  f1-score   support

 Azerbaijani      1.000     0.964     0.982       139
     English      0.988     0.975     0.981        81
     Russian      1.000     1.000     1.000        67

   micro avg      0.996     0.976     0.986       287
   macro avg      0.996     0.980     0.988       287
weighted avg      0.996     0.976     0.986       287
 samples avg      0.997     0.982     0.985       287



In [17]:
for lang in LANGUAGES:
    yt = y_true_lang[lang]
    yp = y_pred_lang[lang]

    true_count = yt.sum()
    pred_count = yp.sum()
    correct    = (yt & yp).sum()

    precision    = correct / pred_count if pred_count > 0 else float("nan")
    recall       = correct / true_count if true_count > 0 else float("nan")
    precision_ci = proportion_confint(correct, pred_count, method="wilson")
    recall_ci    = proportion_confint(correct, true_count, method="wilson")

    print(f"\n{lang}")
    print(f"  precision: {precision:.3f}  CI: {precision_ci}")
    print(f"  recall:    {recall:.3f}  CI: {recall_ci}")


Azerbaijani
  precision: 1.000  CI: (0.9721313249761008, 1.0000000000000002)
  recall:    0.964  CI: (0.9185599908074561, 0.9845391588724067)

English
  precision: 0.988  CI: (0.9325372992233133, 0.9977900244795476)
  recall:    0.975  CI: (0.9143727407780347, 0.9932024125763359)

Russian
  precision: 1.000  CI: (0.9457738606086981, 0.9999999999999998)
  recall:    1.000  CI: (0.9457738606086981, 0.9999999999999998)
